# Problem: Train a 3D CNN network for segmenting CT images
### Problem Statement
You are tasked with employing and evaluating a 3D CNN model in Pytorch for semantic segmentation on synthetically generated CT images. 
Your goal is to review the input and label data shapes. Next, define a MedCNN model class with a `forward` method that emulates a encode-decoder architecture with appropriate input and output channels based on the input shapes.   

### Requirements
1. **Implement** a MedCNN model class with Conv3D and ConvTranspose3d for downsampling and upsampling respectively.
2. **Define** Dice loss for the problem.
2. **Perform** transfer learning from a ResNet18 - a common strategy for custom architectures.
3. **Train** the model for 5 epochs.
### Constraints
- Use `Pytorch` in-built convolution layers
- Ensure, there is a segmentation head at the end of the network


<details>
  <summary>💡 Hint</summary>
  - Strip off the `Avgpooling` and linear layers from ResNet18 using `list(resnet_model.children())[:-2]`
  <br>
  - [Conv3D](https://pytorch.org/docs/stable/generated/torch.nn.Conv3d.html)
  <br>
  - [ConvTranspose3D](https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose3d.html)
  <br>
  - [Forum discussion on model.children](https://discuss.pytorch.org/t/module-children-vs-module-modules/4551)
</details>

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import torch

In [ ]:
# Generate synthetic CT-scan data (batches, slices, RGB) and associated segmentation masks
torch.manual_seed(42)
batch = 100
num_slices = 10
channels = 3
width = 256
height = 256

ct_images = torch.randn(size=(batch, num_slices, channels, width, height))
segmentation_masks = (
    torch.randn(size=(batch, num_slices, 1, width, height)) > 0
).float()

print(f"CT images (train examples) shape: {ct_images.shape}")
print(f"Segmentation binary masks (labels) shape: {segmentation_masks.shape}")

In [ ]:
# Define the MedCNN class and its forward method
class MedCNN(nn.Module):
    def __init__(self, backbone, out_channel=1):
        super(MedCNN, self).__init__()
        self.backbone = backbone

        # TODO: Add Downsample convolutional layers
        ...

        # TODO: Add Upsample convolutional layers
        ...

        # TODO: Final convolution layer from 16 to 1 channel
        ...

        self.relu = nn.ReLU()

    def forward(self, x):
        b, d, c, w, h = x.size()  # Input size: [B, D, C, W, H]
        print(f"Input shape [B, D, C, W, H]: {b, d, c, w, h}")

        # TODO: make changes to the shape of the input such that it is compatible with ResNet
        ...

        # TODO: take output features from the backbone ResNet and make it compatible with Conv3D format
        ...

        # TODO: Downsampling
        ...

        # TODO: Upsampling

        # TODO: final segmentation head
        ...

        return x

In [ ]:
# TODO: define Dice loss
def compute_dice_loss(pred, labels, eps=1e-8):
    """
    Args
    pred: [B, D, 1, W, H]
    labels: [B, D, 1, W, H]

    Returns
    dice_loss: [B, D, 1, W, H]
    """
    ...

In [ ]:
# Define resnet as the backbone removing the last two layers
resnet_model = torchvision.models.resnet18(pretrained=True)
resnet_model = nn.Sequential(*list(resnet_model.children())[:-2])

model = MedCNN(backbone=resnet_model)

optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# TODO: Write training loop
epochs = 5
for epoch in range(epochs):
    optimizer.zero_grad()
    pred = model(ct_images)
    loss = compute_dice_loss(pred, segmentation_masks)
    loss.backward()
    optimizer.step()
    print(f"Loss at epoch {epoch}: {loss}")

In [ ]:
# TODO: Write inference part (with visualization)